# Chapter 6 & 7 Thesis Plots

Core argument figures for the Results and Discussion chapters.

| Figure | Section | Claim proved |
|--------|---------|--------------|
| Fig 6.1 | §6.3 | ZS baseline — pretraining > parameter count |
| Fig 6.2 | §6.4 | Few-shot can hurt (ZS → FS slope) |
| Fig 6.3 | §6.5 | In-context priming > visual grounding (k-sweep) |
| Fig 6.4 | §6.7 | **KEY** — QUR vs FRR calibration scatter |
| Fig 6.5 | §6.8 | C2 > C1 pattern; BDocs C3 = generation artifact |
| Fig 6.6 | §6.9.1 | InternVL gain near-zero even after headroom normalisation |
| Fig 6.7 | §6.9.3 | Location/spatial corruptions consistently hardest (SlideVQA) |
| Fig 7.1 | §7.1 | Discussion anchor — ZS vs FT per model (macro-avg) |


In [ ]:
import os, re, json, glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── Paths ──────────────────────────────────────────────────────────────
BASE       = "/home/amartinelli/VRD-UQA/artifacts/presentations"
SWEEP_DIR  = "/home/amartinelli/VRD-UQA/artifacts/evaluation_runs/eval_val_100_20260617_213228"
OUTPUT_DIR = "output_ch6_ch7"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model registry ──────────────────────────────────────────────────────
MODELS = [
    {"key": "qwen_all_val_300",     "label": "Qwen2.5-VL",  "color": "#4C72B0", "marker": "o"},
    {"key": "internvl_all_val_300", "label": "InternVL3.5", "color": "#DD8452", "marker": "s"},
    {"key": "phi4_all_val_300",     "label": "Phi-4",        "color": "#55A868", "marker": "^"},
    {"key": "gemma4_all_val_300",   "label": "Gemma4",       "color": "#C44E52", "marker": "D"},
]
MODEL_BY_KEY    = {m["key"]: m for m in MODELS}
MODEL_COLORS    = {m["key"]: m["color"]  for m in MODELS}
MODEL_LABELS    = {m["key"]: m["label"]  for m in MODELS}
MODEL_MARKERS   = {m["key"]: m["marker"] for m in MODELS}

# ── Dataset registry ────────────────────────────────────────────────────
DATASETS        = ["DUDE", "MPDocVQA", "SlideVQA", "BDocs"]
DATASET_LABELS  = {"DUDE": "DUDE", "MPDocVQA": "MPDocVQA",
                   "SlideVQA": "SlideVQA", "BDocs": "BoundingDocs"}
DATASET_MARKERS = {"DUDE": "o", "MPDocVQA": "s", "SlideVQA": "^", "BDocs": "D"}

# ── Strategy registry ───────────────────────────────────────────────────
STRATEGIES      = ["zeroshot", "fewshot", "finetuned"]
STRATEGY_LABELS = {
    "zeroshot":  "Zero-Shot",
    "fewshot":   "Few-Shot (k=2, mixed)",
    "finetuned": "Fine-Tuned (LoRA)",
}
STRATEGY_COLORS = {"zeroshot": "#7B2FBE", "fewshot": "#DC3545", "finetuned": "#20B2AA"}

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.linestyle": "--", "grid.alpha": 0.5,
})

# ── Helpers ─────────────────────────────────────────────────────────────
def strategy_key(slug):
    for prefix in ("zeroshot", "fewshot", "finetuned"):
        if slug.startswith(prefix):
            return prefix
    return slug

def discover_slug(model_key, dataset, strategy):
    for p in glob.glob(os.path.join(BASE, model_key, dataset, "*", "manifest.json")):
        slug = os.path.basename(os.path.dirname(p))
        if strategy_key(slug) == strategy:
            return slug
    return None

def load_csv(model_key, dataset, slug, filename):
    path = os.path.join(BASE, model_key, dataset, slug, "metrics", filename)
    if not os.path.exists(path):
        return None
    s = pd.read_csv(path, index_col=0)
    s.columns = ["value"]
    return s["value"]

print("Setup complete. Models:", [m["label"] for m in MODELS])


In [ ]:
records = []
for m in MODELS:
    for dataset in DATASETS:
        for strat in STRATEGIES:
            slug = discover_slug(m["key"], dataset, strat)
            if slug is None:
                continue
            qur_s = load_csv(m["key"], dataset, slug, "QUR.csv")
            frr_s = load_csv(m["key"], dataset, slug, "FRR.csv")
            if qur_s is None:
                continue
            # BDocs has no C3 questions — QUR_C3=0.0 is a generation artifact
            c3_val = qur_s.get("QUR_C3", np.nan)
            if c3_val == 0.0 and dataset == "BDocs":
                c3_val = np.nan
            records.append({
                "model":       m["key"],
                "model_label": m["label"],
                "dataset":     dataset,
                "strategy":    strat,
                "QUR":         qur_s.get("QUR",    np.nan),
                "QUR_C1":      qur_s.get("QUR_C1", np.nan),
                "QUR_C2":      qur_s.get("QUR_C2", np.nan),
                "QUR_C3":      c3_val,
                "FRR":         frr_s.get("FRR",    np.nan) if frr_s is not None else np.nan,
                "FRR_C1":      frr_s.get("FRR_C1", np.nan) if frr_s is not None else np.nan,
                "FRR_C2":      frr_s.get("FRR_C2", np.nan) if frr_s is not None else np.nan,
            })

df = pd.DataFrame(records)
print(f"Loaded {len(df)} records")
df.pivot_table(index=["model_label", "strategy"], columns="dataset", values="QUR").round(3)


## Fig 6.1 — Zero-Shot QUR by Model and Dataset (§6.3)

Baseline: pretraining strategy matters more than parameter count. Phi-4 SlideVQA=0.16 is the lowest single result in the study.

In [ ]:
zs_df = df[df.strategy == "zeroshot"]

n_models  = len(MODELS)
bar_w     = 0.2
group_gap = 0.1
group_w   = n_models * bar_w + group_gap
x_groups  = np.arange(len(DATASETS)) * group_w

fig, ax = plt.subplots(figsize=(10, 5))

for i, m in enumerate(MODELS):
    sub  = zs_df[zs_df.model == m["key"]]
    vals = [sub.loc[sub.dataset == d, "QUR"].values[0]
            if not sub[sub.dataset == d].empty else np.nan for d in DATASETS]
    x_pos     = x_groups + i * bar_w
    plot_vals = [v if not np.isnan(v) else 0 for v in vals]
    bars = ax.bar(x_pos, plot_vals, width=bar_w,
                  color=m["color"], label=m["label"],
                  edgecolor="white", linewidth=0.6)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.013,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=8)
        else:
            bar.set_facecolor("#dddddd")
            bar.set_edgecolor("#aaaaaa")
            bar.set_hatch("///")

tick_pos = x_groups + (n_models - 1) * bar_w / 2
ax.set_xticks(tick_pos)
ax.set_xticklabels([DATASET_LABELS[d] for d in DATASETS], fontsize=11)
ax.set_ylabel("Question Unanswerability Rate (QUR)", fontsize=11)
ax.set_ylim(0, 1.1)
ax.axhline(0.5, color="grey", linestyle=":", linewidth=1, alpha=0.5)
ax.set_title(
    "Fig 6.1 — Zero-Shot QUR by Model and Dataset\n"
    "(pretraining strategy matters more than parameter count)",
    fontsize=12, fontweight="bold", pad=10,
)
ax.legend(fontsize=10, framealpha=0.85)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig6_1_zeroshot_qur_grouped_bar.pdf"), bbox_inches="tight")
plt.show()


## Fig 6.2 — Zero-Shot → Few-Shot Slope (§6.4)

Downward slopes show few-shot hurts. Key examples: InternVL MPDocVQA (0.630→0.513), Phi-4 MPDocVQA (0.397→0.313).

In [ ]:
def _stagger_ys(ys, min_gap=0.07):
    """Bidirectional stagger: forward pass pushes up, backward pass re-centres cluster."""
    ys = list(ys)
    for i in range(1, len(ys)):
        if ys[i] - ys[i - 1] < min_gap:
            ys[i] = ys[i - 1] + min_gap
    for i in range(len(ys) - 2, -1, -1):
        if ys[i + 1] - ys[i] > min_gap:
            ys[i] = min(ys[i], ys[i + 1] - min_gap)
    return ys


fig, axes = plt.subplots(1, len(DATASETS), figsize=(14, 5), sharey=True)

for ax_i, dataset in enumerate(DATASETS):
    ax = axes[ax_i]
    anns = []

    for m in MODELS:
        zs_row = df[(df.model == m["key"]) & (df.dataset == dataset) & (df.strategy == "zeroshot")]
        fs_row = df[(df.model == m["key"]) & (df.dataset == dataset) & (df.strategy == "fewshot")]
        if zs_row.empty or fs_row.empty:
            continue
        zs_val = zs_row["QUR"].values[0]
        fs_val = fs_row["QUR"].values[0]
        if np.isnan(zs_val) or np.isnan(fs_val):
            continue
        ax.plot([0, 1], [zs_val, fs_val],
                color=m["color"], linewidth=2, alpha=0.85,
                marker="o", markersize=8,
                markeredgecolor="white", markeredgewidth=1)
        anns.append({"mid_y": (zs_val + fs_val) / 2, "delta": fs_val - zs_val, "color": m["color"]})

    anns.sort(key=lambda a: a["mid_y"])
    label_ys = _stagger_ys([a["mid_y"] for a in anns])

    for ann, ly in zip(anns, label_ys):
        moved = abs(ly - ann["mid_y"]) > 0.015
        ax.annotate(
            f"{ann['delta']:+.2f}",
            xy=(0.5, ann["mid_y"]),
            xytext=(0.56, ly),
            xycoords="data", textcoords="data",
            arrowprops=dict(arrowstyle="-", color=ann["color"], lw=0.7, alpha=0.5) if moved else None,
            fontsize=7.5, color=ann["color"], fontweight="bold",
            ha="left", va="center",
        )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Zero-Shot", "Few-Shot\n(k=2 mixed)"], fontsize=9)
    ax.set_xlim(-0.25, 1.35)
    ax.set_ylim(0.0, 1.05)
    ax.set_title(DATASET_LABELS[dataset], fontsize=11, fontweight="bold")
    if ax_i == 0:
        ax.set_ylabel("QUR", fontsize=10)
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)

handles = [
    Line2D([0], [0], color=m["color"], linewidth=2.5,
           marker="o", markersize=8, markeredgecolor="white",
           label=m["label"])
    for m in MODELS
]
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=10,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle(
    "Fig 6.2 — Zero-Shot → Few-Shot QUR Trajectory (Δ annotated on each segment)\n"
    "Downward slopes = few-shot hurts; mixed-signal effect from 1 answerable + 1 unanswerable demo",
    fontsize=11, fontweight="bold",
)
plt.tight_layout(rect=[0, 0.09, 1, 1])
plt.savefig(os.path.join(OUTPUT_DIR, "fig6_2_zs_to_fs_slope.pdf"), bbox_inches="tight")
plt.show()


## Fig 6.3 — k-Sweep: Answerable-Only vs Mixed-Random (§6.5)

Qwen2.5-VL-7B only, SlideVQA + BDocs. Counter-intuitive result: answerable-only demos *reduce* QUR — in-context priming dominates visual-grounding signal.

In [ ]:
# ── Load k-sweep data ───────────────────────────────────────────────────
SWEEP_DATASETS = ["SlideVQA", "BDocs"]

def parse_sweep_slug(slug):
    m = re.match(r"fewshot_ocr_k(\d+)_(?:mixed|answerable)_(random|handpicked|specific)", slug)
    return (int(m.group(1)), m.group(2)) if m else (None, None)

def load_sweep_csv(dataset, slug, filename):
    path = os.path.join(SWEEP_DIR, dataset, slug, "metrics", filename)
    if not os.path.exists(path):
        return None
    s = pd.read_csv(path, index_col=0)
    s.columns = ["value"]
    return s["value"]

sweep_configs = {}
for ds in SWEEP_DATASETS:
    sweep_configs[ds] = {}
    for p in sorted(glob.glob(os.path.join(SWEEP_DIR, ds, "*", "manifest.json"))):
        slug = os.path.basename(os.path.dirname(p))
        k, sel = parse_sweep_slug(slug)
        if k is not None:
            sweep_configs[ds][slug] = {"k": k, "sel": sel}

sweep_records = []
for ds in SWEEP_DATASETS:
    for slug, info in sweep_configs[ds].items():
        s = load_sweep_csv(ds, slug, "QUR.csv")
        if s is None:
            continue
        c3 = s.get("QUR_C3", np.nan)
        if c3 == 0.0:
            c3 = np.nan
        sweep_records.append({
            "dataset": ds, "config": slug,
            "k": info["k"], "sel": info["sel"],
            "QUR":    s.get("QUR",    np.nan),
            "QUR_C1": s.get("QUR_C1", np.nan),
            "QUR_C2": s.get("QUR_C2", np.nan),
            "QUR_C3": c3,
        })

sweep_df = pd.DataFrame(sweep_records)
print(sweep_df[["dataset", "k", "sel", "QUR"]].to_string(index=False))


In [ ]:
K_VALS        = [2, 4, 6]
COLOR_SWEEP   = "#4C72B0"   # answerable-only handpicked
COLOR_RANDOM  = "#DD8452"   # mixed-random baseline

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

for idx, ds in enumerate(SWEEP_DATASETS):
    ax   = axes[idx]
    sub  = sweep_df[sweep_df.dataset == ds]
    hp   = sub[sub.sel != "random"].sort_values("k")
    rand = sub[sub.sel == "random"]

    if not hp.empty:
        ax.plot(hp["k"], hp["QUR"],
                marker="o", color=COLOR_SWEEP, linewidth=2.5, markersize=9,
                label="Answerable-only (handpicked)", zorder=3)
        for _, row in hp.iterrows():
            ax.annotate(f"{row['QUR']:.2f}", (row["k"], row["QUR"]),
                        textcoords="offset points", xytext=(0, 10),
                        ha="center", fontsize=9, color=COLOR_SWEEP, fontweight="bold")

    if not rand.empty:
        rv = rand["QUR"].values[0]
        ax.axhline(rv, color=COLOR_RANDOM, linestyle="--", linewidth=2.2,
                   label=f"Mixed-random k=2 baseline ({rv:.2f})", zorder=2)
        ax.annotate(f"{rv:.2f}", xy=(K_VALS[-1], rv),
                    textcoords="offset points", xytext=(8, 5),
                    ha="left", fontsize=9, color=COLOR_RANDOM, fontweight="bold")

    ax.set_xticks(K_VALS)
    ax.set_xlabel("k (few-shot demos)", fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_title(DATASET_LABELS[ds], fontsize=12, fontweight="bold")
    if idx == 0:
        ax.set_ylabel("QUR  (Qwen2.5-VL)", fontsize=10)
    ax.legend(fontsize=9, loc="upper right")

fig.suptitle(
    "Fig 6.3 — k-Sweep: Answerable-Only Demos vs Mixed-Random Baseline (Qwen2.5-VL)\n"
    "Answerable-only reduces QUR — in-context output priming dominates visual grounding",
    fontsize=11, fontweight="bold",
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig6_3_ksweep_priming_vs_grounding.pdf"), bbox_inches="tight")
plt.show()


## Fig 6.4 — QUR vs FRR Calibration Scatter (§6.7) — KEY PLOT

One point per (model × dataset), fine-tuned condition only (FRR not computed for ZS/FS). Top-left = good calibration. Diagonal = indiscriminate refusal.

In [ ]:
ft_df = df[(df.strategy == "finetuned") & df["QUR"].notna() & df["FRR"].notna()].copy()
model_means = ft_df.groupby("model")[["FRR", "QUR"]].mean()

# Fixed annotation anchors (axes-fraction coords) — chosen to avoid overlap
STAR_ANCHORS = {
    "qwen_all_val_300":     (0.97, 0.97),
    "gemma4_all_val_300":   (0.04, 0.97),
    "phi4_all_val_300":     (0.04, 0.55),
    "internvl_all_val_300": (0.97, 0.38),
}
STAR_HA = {
    "qwen_all_val_300": "right", "gemma4_all_val_300": "left",
    "phi4_all_val_300": "left",  "internvl_all_val_300": "right",
}

fig, ax = plt.subplots(figsize=(8, 7))

for m in MODELS:
    m_sub = ft_df[ft_df.model == m["key"]]
    for _, row in m_sub.iterrows():
        ax.scatter(row["FRR"], row["QUR"],
                   color=m["color"], marker=DATASET_MARKERS[row["dataset"]],
                   s=130, zorder=4, edgecolors="white", linewidths=1.0, alpha=0.90)

for m in MODELS:
    if m["key"] not in model_means.index:
        continue
    mx = model_means.loc[m["key"], "FRR"]
    my = model_means.loc[m["key"], "QUR"]
    ax.scatter(mx, my, color=m["color"], marker="*", s=440, zorder=5,
               edgecolors="black", linewidths=0.9)
    ax.annotate(
        m["label"], xy=(mx, my), xycoords="data",
        xytext=STAR_ANCHORS[m["key"]], textcoords="axes fraction",
        ha=STAR_HA[m["key"]], va="center",
        fontsize=9.5, fontweight="bold", color=m["color"],
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor=m["color"], linewidth=1.2, alpha=0.92),
        arrowprops=dict(arrowstyle="-|>", color=m["color"],
                        lw=1.4, connectionstyle="arc3,rad=0.15"),
    )

ax.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1, alpha=0.4, zorder=1)
ax.scatter([0], [1], marker="*", s=450, color="gold", edgecolors="black",
           linewidths=0.9, zorder=6)
ax.annotate("Ideal", xy=(0, 1), xytext=(14, -14),
            textcoords="offset points", fontsize=9, color="#888800", fontweight="bold")

ax.axhline(0.5, color="#bbbbbb", linestyle=":", linewidth=0.9)
ax.axvline(0.2, color="#bbbbbb", linestyle=":", linewidth=0.9)

model_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=m["color"],
           markersize=10, label=m["label"])
    for m in MODELS
]
ds_handles = [
    Line2D([0], [0], marker=DATASET_MARKERS[d], color="grey",
           markersize=9, markeredgecolor="white", label=DATASET_LABELS[d])
    for d in DATASETS
]
star_h = Line2D([0], [0], marker="*", color="w", markerfacecolor="grey",
                markersize=13, markeredgecolor="black", label="Model avg (\u2605)")
diag_h = Line2D([0], [0], color="grey", linestyle="--",
                linewidth=1, alpha=0.6, label="Collapse diagonal")
ax.legend(handles=model_handles + [star_h] + ds_handles + [diag_h],
          fontsize=8.5, loc="lower right", framealpha=0.90, ncol=2)

ax.set_xlim(-0.04, 0.60)
ax.set_ylim(0.35, 1.08)
ax.set_xlabel("False Refusal Rate (FRR)  — incorrectly refuses answerable  \u2193 better",
              fontsize=10)
ax.set_ylabel("Question Unanswerability Rate (QUR)  — correctly refuses unanswerable  \u2191 better",
              fontsize=10)
ax.set_title(
    "Fig 6.4 — Refusal Calibration: QUR vs FRR (Fine-Tuned LoRA)\n"
    "Top-left = discriminates well  |  Diagonal = collapses to fixed refusal rate",
    fontsize=12, fontweight="bold", pad=12,
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig6_4_qur_vs_frr_calibration_scatter.pdf"), bbox_inches="tight")
plt.show()


## Fig 6.5 — Complexity Heatmap: Fine-Tuned QUR (§6.8)

C2 > C1 across all models and datasets. BDocs C3 = generation artifact (gray). SlideVQA C3 values available.

In [ ]:
ft_sub      = df[df.strategy == "finetuned"]
COMP_COLS   = ["QUR_C1", "QUR_C2", "QUR_C3"]
COMP_LABELS = ["C1", "C2", "C3"]

col_headers = [f"{DATASET_LABELS[d]}\n{c}" for d in DATASETS for c in COMP_LABELS]
n_rows = len(MODELS)
n_cols = len(DATASETS) * len(COMP_COLS)

heat        = np.full((n_rows, n_cols), np.nan)
is_artifact = np.zeros((n_rows, n_cols), dtype=bool)

for r, m in enumerate(MODELS):
    col_idx = 0
    for d in DATASETS:
        row = ft_sub[(ft_sub.model == m["key"]) & (ft_sub.dataset == d)]
        for cx in COMP_COLS:
            if not row.empty:
                heat[r, col_idx] = row[cx].values[0]
            if d == "BDocs" and cx == "QUR_C3":
                is_artifact[r, col_idx] = True
            col_idx += 1

# Mask artifact cells from colormap
display_heat = heat.copy()
display_heat[is_artifact] = np.nan

fig, ax = plt.subplots(figsize=(n_cols * 1.45 + 1, n_rows * 1.0 + 2.0))
im = ax.imshow(display_heat, cmap="YlGn", vmin=0, vmax=1, aspect="auto")

# Artifact cells — distinct gray background
for r in range(n_rows):
    for c in range(n_cols):
        if is_artifact[r, c]:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                       facecolor="#d0d0d0", zorder=1))

# Dataset separator lines
n_cx = len(COMP_COLS)
for sep in range(1, len(DATASETS)):
    ax.axvline(sep * n_cx - 0.5, color="white", linewidth=3, zorder=3)

ax.set_xticks(range(n_cols))
ax.set_xticklabels(col_headers, fontsize=9)
ax.set_yticks(range(n_rows))
ax.set_yticklabels([m["label"] for m in MODELS], fontsize=10.5)

for r in range(n_rows):
    for c in range(n_cols):
        if is_artifact[r, c]:
            ax.text(c, r, "artifact\n(\u00a74.6)", ha="center", va="center",
                    fontsize=7, color="#555555", zorder=4)
        elif np.isnan(heat[r, c]):
            ax.text(c, r, "N/A", ha="center", va="center",
                    fontsize=8, color="#aaaaaa", zorder=4)
        else:
            v  = heat[r, c]
            tc = "white" if v > 0.75 else "black"
            ax.text(c, r, f"{v:.2f}", ha="center", va="center",
                    fontsize=10, fontweight="bold", color=tc, zorder=4)

plt.colorbar(im, ax=ax, label="Fine-Tuned QUR", shrink=0.75)
ax.set_title(
    "Fig 6.5 — Fine-Tuned QUR by Model, Dataset and Complexity Level\n"
    "C2 > C1 pattern across all models  |  gray = BDocs C3 generation artifact (\u00a74.6)",
    fontsize=11, fontweight="bold", pad=14,
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig6_5_complexity_heatmap_ft_qur.pdf"), bbox_inches="tight")
plt.show()


## Fig 6.6 — Headroom-Normalised ΔQ per Model (§6.9.1)

Normalisation formula: ΔQ / (1 − QUR_ZS). Proves InternVL's near-zero gain is **architectural**, not a ceiling effect. Macro-averaged across all 4 datasets.

In [ ]:
norm_records = []
for m in MODELS:
    zs_qurs, ft_qurs = [], []
    for d in DATASETS:
        zs_r = df[(df.model == m["key"]) & (df.dataset == d) & (df.strategy == "zeroshot")]
        ft_r = df[(df.model == m["key"]) & (df.dataset == d) & (df.strategy == "finetuned")]
        if not zs_r.empty and not ft_r.empty:
            zq = zs_r["QUR"].values[0]
            fq = ft_r["QUR"].values[0]
            if not (np.isnan(zq) or np.isnan(fq)):
                zs_qurs.append(zq)
                ft_qurs.append(fq)
    if not zs_qurs:
        continue
    zs_avg   = np.mean(zs_qurs)
    ft_avg   = np.mean(ft_qurs)
    delta    = ft_avg - zs_avg
    headroom = 1.0 - zs_avg
    norm_records.append({
        "model":      m["key"],
        "label":      m["label"],
        "color":      m["color"],
        "ZS_avg":     zs_avg,
        "FT_avg":     ft_avg,
        "delta_abs":  delta,
        "norm_delta": delta / headroom if headroom > 0 else np.nan,
    })

norm_df = pd.DataFrame(norm_records)
print(norm_df[["label", "ZS_avg", "FT_avg", "delta_abs", "norm_delta"]].round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = norm_df["label"].values
colors = norm_df["color"].values

# Left — absolute ΔQ
ax = axes[0]
bars = ax.bar(labels, norm_df["delta_abs"], color=colors, edgecolor="white", linewidth=0.7)
for bar, val in zip(bars, norm_df["delta_abs"].values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"+{val:.3f}", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Absolute \u0394Q  (FT \u2212 ZS, macro-avg)", fontsize=10)
ax.set_ylim(0, 0.38)
ax.set_title("Absolute QUR Gain", fontsize=11, fontweight="bold")

# Right — headroom-normalised ΔQ
ax = axes[1]
bars = ax.bar(labels, norm_df["norm_delta"], color=colors, edgecolor="white", linewidth=0.7)
for bar, val in zip(bars, norm_df["norm_delta"].values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Headroom-Normalised \u0394Q = \u0394Q / (1 \u2212 QUR_ZS)", fontsize=10)
ax.set_ylim(0, 0.70)
ax.set_title("Headroom-Normalised QUR Gain\n(rules out ceiling effect)",
             fontsize=11, fontweight="bold")

fig.suptitle(
    "Fig 6.6 — Fine-Tuning Gain per Model (macro-avg, 4 datasets)\n"
    "InternVL near-zero gap persists after normalisation \u2192 architectural, not ceiling",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig6_6_headroom_normalised_delta_q.pdf"), bbox_inches="tight")
plt.show()


## Fig 6.7 — SlideVQA: Spatial/Location Corruptions Consistently Hardest (§6.9.3)

QUR by NLPE entity type for SlideVQA across all 4 models (ZS and FT). LOCATION entity type maps to spatial/layout corruptions and scores lowest in every condition. Model answers from visual content without verifying the spatial anchor.

In [ ]:
ENTITY_TYPES = ["NUMERIC", "TEMPORAL", "ENTITY", "LOCATION", "STRUCTURE"]
ET_LABELS    = {
    "NUMERIC":   "Numeric",
    "TEMPORAL":  "Temporal",
    "ENTITY":    "Entity",
    "LOCATION":  "Location\n(spatial/layout)",
    "STRUCTURE": "Structure",
}

nlpe_records = []
for m in MODELS:
    for strat in ["zeroshot", "finetuned"]:
        slug = discover_slug(m["key"], "SlideVQA", strat)
        if slug is None:
            continue
        s = load_csv(m["key"], "SlideVQA", slug, "QUR_NLPE.csv")
        if s is None:
            continue
        for et in ENTITY_TYPES:
            nlpe_records.append({
                "model":    m["key"],
                "strategy": strat,
                "entity":   et,
                "QUR":      s.get(et, np.nan),
            })

nlpe_df = pd.DataFrame(nlpe_records)

bar_w = 0.18
x     = np.arange(len(ENTITY_TYPES))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax_i, strat in enumerate(["zeroshot", "finetuned"]):
    ax  = axes[ax_i]
    sub = nlpe_df[nlpe_df.strategy == strat]

    # Shade the LOCATION column
    loc_idx = ENTITY_TYPES.index("LOCATION")
    ax.axvspan(loc_idx - 0.1, loc_idx + len(MODELS) * bar_w + 0.1,
               alpha=0.08, color="red", zorder=0)

    for i, m in enumerate(MODELS):
        m_sub = sub[sub.model == m["key"]].set_index("entity")
        vals  = [m_sub.loc[et, "QUR"] if et in m_sub.index else np.nan
                 for et in ENTITY_TYPES]
        ax.bar(x + i * bar_w, vals, width=bar_w,
               color=m["color"], label=m["label"],
               edgecolor="white", linewidth=0.6)

    ax.set_xticks(x + (len(MODELS) - 1) * bar_w / 2)
    ax.set_xticklabels([ET_LABELS[e] for e in ENTITY_TYPES], fontsize=9)
    ax.set_ylim(0, 1.1)
    if ax_i == 0:
        ax.set_ylabel("QUR — SlideVQA", fontsize=10)
    ax.set_title(STRATEGY_LABELS[strat], fontsize=11, fontweight="bold")
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9, alpha=0.5)

handles = [mpatches.Patch(color=m["color"], label=m["label"]) for m in MODELS]
shade_h = mpatches.Patch(facecolor="red", alpha=0.15,
                          label="Location / spatial (shaded)")
fig.legend(handles=handles + [shade_h], loc="lower center", ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, -0.04))
fig.suptitle(
    "Fig 6.7 — SlideVQA QUR by NLPE Entity Type\n"
    "Location/spatial corruptions score lowest across all models and conditions",
    fontsize=12, fontweight="bold",
)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(OUTPUT_DIR, "fig6_7_slidevqa_location_spatial_lowest.pdf"), bbox_inches="tight")
plt.show()


## Fig 7.1 — ZS vs Fine-Tuned QUR per Model (§7.1 discussion anchor)

Macro-averaged across 4 datasets. Fine-tuning substantially helps 3 of 4 models; InternVL gain near-zero. ΔQ arrow annotated between ZS and FT bar per model.

In [ ]:
macro_df = (
    df[df.strategy.isin(["zeroshot", "finetuned"])]
    .groupby(["model", "model_label", "strategy"])["QUR"]
    .mean()
    .reset_index()
)

n_models  = len(MODELS)
bar_w     = 0.3
group_gap = 0.18
group_w   = 2 * bar_w + group_gap
x_groups  = np.arange(n_models) * group_w

fig, ax = plt.subplots(figsize=(10, 5.5))

for s_idx, strat in enumerate(["zeroshot", "finetuned"]):
    sub   = macro_df[macro_df.strategy == strat]
    alpha = 0.55 if strat == "zeroshot" else 1.0
    for m_idx, m in enumerate(MODELS):
        row = sub[sub.model == m["key"]]
        val = row["QUR"].values[0] if not row.empty else np.nan
        if np.isnan(val):
            continue
        x_pos = x_groups[m_idx] + s_idx * bar_w
        ax.bar(x_pos, val, width=bar_w,
               color=STRATEGY_COLORS[strat], alpha=alpha,
               edgecolor="white", linewidth=0.7)
        ax.text(x_pos + bar_w / 2, val + 0.012,
                f"{val:.2f}", ha="center", va="bottom",
                fontsize=9, fontweight="bold", color=STRATEGY_COLORS[strat])

# ΔQ arrows and annotations
for m_idx, m in enumerate(MODELS):
    zs_row = macro_df[(macro_df.model == m["key"]) & (macro_df.strategy == "zeroshot")]
    ft_row = macro_df[(macro_df.model == m["key"]) & (macro_df.strategy == "finetuned")]
    if zs_row.empty or ft_row.empty:
        continue
    zs_v = zs_row["QUR"].values[0]
    ft_v = ft_row["QUR"].values[0]
    delta = ft_v - zs_v
    x_zs  = x_groups[m_idx] + 0 * bar_w + bar_w / 2
    x_ft  = x_groups[m_idx] + 1 * bar_w + bar_w / 2
    mid_y = (zs_v + ft_v) / 2 + 0.03
    ax.annotate("", xy=(x_ft, ft_v + 0.01), xytext=(x_zs, zs_v + 0.01),
                arrowprops=dict(arrowstyle="-|>", color="#555555",
                               lw=1.2, connectionstyle="arc3,rad=-0.2"))
    ax.text((x_zs + x_ft) / 2, mid_y + 0.02,
            f"\u0394{delta:+.2f}", ha="center", fontsize=8.5,
            color="#333333", fontweight="bold")

tick_pos = x_groups + bar_w / 2
ax.set_xticks(tick_pos)
ax.set_xticklabels([m["label"] for m in MODELS], fontsize=11)
ax.set_ylabel("QUR (macro-averaged across 4 datasets)", fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color="grey", linestyle=":", linewidth=1, alpha=0.5)

handles = [
    mpatches.Patch(color=STRATEGY_COLORS["zeroshot"],  alpha=0.55, label="Zero-Shot"),
    mpatches.Patch(color=STRATEGY_COLORS["finetuned"],             label="Fine-Tuned (LoRA)"),
]
ax.legend(handles=handles, fontsize=10, framealpha=0.85, loc="upper left")
ax.set_title(
    "Fig 7.1 — Zero-Shot vs Fine-Tuned QUR per Model (macro-avg, 4 datasets)\n"
    "Fine-tuning substantially improves 3 of 4 models; InternVL gain near-zero",
    fontsize=12, fontweight="bold", pad=12,
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig7_1_zs_vs_ft_summary_macro.pdf"), bbox_inches="tight")
plt.show()
